In [1]:
import pandas as pd
import os
import ast

In [2]:
# Define relative paths assuming this notebook is inside the /notebooks folder
ml25m_dir = "../data/ml-25m/"
tmdb_dir = "../data/TMDB Metadata/"

print("Loading datasets into memory... (This might take a moment due to the file sizes)")

# 1. Load MovieLens Data
movies = pd.read_csv(os.path.join(ml25m_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(ml25m_dir, 'ratings.csv'))
links = pd.read_csv(os.path.join(ml25m_dir, 'links.csv'))

# 2. Load TMDB Data
# low_memory=False prevents mixed-type warnings on massive files
tmdb_meta = pd.read_csv(os.path.join(tmdb_dir, 'movies_metadata.csv'), low_memory=False)
keywords = pd.read_csv(os.path.join(tmdb_dir, 'keywords.csv'))
credits = pd.read_csv(os.path.join(tmdb_dir, 'credits.csv'))

print("Success! All CSVs loaded.")

Loading datasets into memory... (This might take a moment due to the file sizes)
Success! All CSVs loaded.


In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [5]:
tmdb_meta.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [19]:
keywords.style.set_properties(subset=['keywords'], **{'min-width': '300px'})
keywords.head()

,id,keywords,keywords_clean
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...",jealousy toy boy friendship friends rivalry bo...
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1...",board game disappearance based on children's b...
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...",fishing best friend duringcreditsstinger old men
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':...",based on novel interracial relationship single...
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...",baby midlife crisis confidence aging daughter ...


In [14]:
credits.head()

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


In [16]:
print("Cleaning data and parsing JSON strings...")

# Filter out bad IDs and unreleased movies from TMDB metadata
tmdb_meta = tmdb_meta[pd.to_numeric(tmdb_meta['id'], errors='coerce').notnull()]
tmdb_meta['id'] = tmdb_meta['id'].astype(int)
tmdb_meta = tmdb_meta[tmdb_meta['status'] == 'Released']

# Helper function to extract names from the JSON strings
def extract_names(x):
    try:
        return " ".join([d['name'] for d in ast.literal_eval(x)])
    except:
        return ""

# Helper function to extract Top 3 Cast members
def extract_top_3_cast(x):
    try:
        cast_list = ast.literal_eval(x)
        # We strip spaces in names (e.g., 'Keanu Reeves' -> 'KeanuReeves') 
        # so the TF-IDF vectorizer treats the full name as a single unique token.
        return " ".join([d['name'].replace(" ", "") for d in cast_list[:3]])
    except:
        return ""

# Helper function to extract the Director
def extract_director(x):
    try:
        crew_list = ast.literal_eval(x)
        for d in crew_list:
            if d['job'] == 'Director':
                return d['name'].replace(" ", "")
        return ""
    except:
        return ""

# Apply the parsing functions
tmdb_meta['genres_clean'] = tmdb_meta['genres'].apply(extract_names)

keywords['id'] = keywords['id'].astype(int)
keywords['keywords_clean'] = keywords['keywords'].apply(extract_names)

credits['id'] = credits['id'].astype(int)
credits['cast_clean'] = credits['cast'].apply(extract_top_3_cast)
credits['director_clean'] = credits['crew'].apply(extract_director)

print("Parsing complete.")

Cleaning data and parsing JSON strings...
Parsing complete.


In [21]:
print("Merging datasets...")

# 1. Stitch TMDB metadata together
tmdb_full = tmdb_meta[['id', 'original_title', 'overview', 'genres_clean']].merge(keywords[['id', 'keywords_clean']], on='id')
tmdb_full = tmdb_full.merge(credits[['id', 'cast_clean', 'director_clean']], on='id')

# 2. Prepare the bridge (links.csv)
links = links[links['tmdbId'].notnull()]
links['tmdbId'] = links['tmdbId'].astype(int)

# 3. Bridge MovieLens and TMDB
movies_with_links = movies.merge(links[['movieId', 'tmdbId']], on='movieId')
master_df = movies_with_links.merge(tmdb_full, left_on='tmdbId', right_on='id')

# 4. Create the final "soup" for our NLP vectorizer
master_df['combined_features'] = master_df['genres_clean'] + " " + master_df['keywords_clean'] + " " + master_df['cast_clean'] + " " + master_df['director_clean']

# Drop columns we no longer need to save memory
master_df = master_df[['movieId', 'original_title', 'overview', 'combined_features']]

print(f"Master DataFrame successfully built! Total movies aligned: {len(master_df)}")

Merging datasets...
Master DataFrame successfully built! Total movies aligned: 43794


In [23]:
import os

print("Exporting processed artifacts...")

# Create a 'processed' directory if it doesn't exist
processed_dir = "../data/processed/"
os.makedirs(processed_dir, exist_ok=True)

# Export the clean metadata matrix
master_df.to_csv(os.path.join(processed_dir, 'master_metadata.csv'), index=False)

# Export a smaller sample of the ratings matrix for fast model prototyping
# (Using 500k rows for now so we can test the SVD algorithm quickly)
ratings_sample = ratings.head(500000) 
ratings_sample.to_csv(os.path.join(processed_dir, 'ratings_sample.csv'), index=False)

print("Data ingestion complete. Clean artifacts saved to /data/processed/")

Exporting processed artifacts...
Data ingestion complete. Clean artifacts saved to /data/processed/
